# Data Integration Phase

Thin runner for gold integration. Analytical queries are exploratory.

```bash
python -m src.jobs.run_gold --stage integrate
```

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, ensure_runtime

ensure_runtime()

from src.lake import GOLD, read_delta, show_delta
from src.gold.integrate import integrate
from src.benchmark.storage import compare_integrated_layouts
from src.queries.analytical import (
    QUERY_1_MONTHLY_ZONE_DEMAND,
    QUERY_2_WEATHER_DISTANCE,
    QUERY_3_PM25_DEMAND,
    QUERY_4_ZONE_WEATHER_SENSITIVITY,
    QUERY_5_PEAK_HOURS_BY_DOW,
    QUERY_6_MONTHLY_TRENDS,
)

spark = create_spark("integration-notebook")

:: loading settings :: url = jar:file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/juozas/.ivy2/cache
The jars for the packages stored in: /Users/juozas/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c46495bb-cfc9-4b97-9751-86f6e8a25e21;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
:: resolution report :: resolve 103ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default   

## Build `integrated_taxi_trips`

In [2]:
integrate(spark)
compare_integrated_layouts()

26/09/24 16:05:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


write by_date: 11.7s
integrated_taxi_trips: 9417383 rows @ data/lake/gold/integrated_taxi_trips
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+-----------------------+--------------+-------------------+-------------------+---------------+-----------+----------+------------+------------+-------------+-----------------+------------------+---------------------------+-----------+-----------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|pickup_location_id|pickup_zone            |pickup_borough|dropoff_location_id|dropoff_zone       |dropoff_borough|fare_amount|tip_amount|tolls_amount|total_amount|temperature_c|wind_speed_ms    |pm25              |pm25_unit                  |pickup_date|pickup_hour|
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+-----------------------+--------------+-------------------+-------------------

write by_borough: 9.3s
Storage
integrated_taxi_trips                    files=  222  partitions=  92  size=   227.7 MB
integrated_taxi_trips_by_borough         files=  200  partitions=   8  size=   226.9 MB


## Exploratory analytical queries (Q1–Q6)

In [3]:
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView(
    "integrated_taxi_trips"
)

for name, sql in [
    ("Q1 monthly zone demand", QUERY_1_MONTHLY_ZONE_DEMAND),
    ("Q2 weather distance", QUERY_2_WEATHER_DISTANCE),
    ("Q3 pm25 demand", QUERY_3_PM25_DEMAND),
    ("Q4 zone weather sensitivity", QUERY_4_ZONE_WEATHER_SENSITIVITY),
    ("Q5 peak hours by DOW", QUERY_5_PEAK_HOURS_BY_DOW),
    ("Q6 monthly trends", QUERY_6_MONTHLY_TRENDS),
]:
    print(f"\n=== {name} ===")
    spark.sql(sql).show(20, truncate=False)


=== Q1 monthly zone demand ===


+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|trip_month|pickup_location_id|pickup_borough|pickup_zone                 |total_trips|active_days|avg_daily_trips|
+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|2024-01-01|161               |Manhattan     |Midtown Center              |141738     |31         |4572.19        |
|2024-01-01|237               |Manhattan     |Upper East Side South       |141263     |31         |4556.87        |
|2024-01-01|132               |Queens        |JFK Airport                 |141159     |31         |4553.52        |
|2024-01-01|236               |Manhattan     |Upper East Side North       |135334     |31         |4365.61        |
|2024-01-01|162               |Manhattan     |Midtown East                |105466     |31         |3402.13        |
|2024-01-01|230               |Manhattan     |Times Sq/Theatre District 

+-----------------------+-----------------------+----------+------------------+
|temp_category          |wind_category          |trip_count|avg_distance_miles|
+-----------------------+-----------------------+----------+------------------+
|Cold (0°C to 10°C)     |Calm (<2 m/s)          |395811    |3.29              |
|Cold (0°C to 10°C)     |High Wind (>6 m/s)     |2272210   |3.31              |
|Cold (0°C to 10°C)     |Moderate Wind (2-6 m/s)|4128105   |3.29              |
|Freezing (<0°C)        |Calm (<2 m/s)          |13866     |4.2               |
|Freezing (<0°C)        |High Wind (>6 m/s)     |478247    |3.23              |
|Freezing (<0°C)        |Moderate Wind (2-6 m/s)|622618    |3.2               |
|Moderate (10°C to 20°C)|Calm (<2 m/s)          |117228    |3.37              |
|Moderate (10°C to 20°C)|High Wind (>6 m/s)     |460189    |3.28              |
|Moderate (10°C to 20°C)|Moderate Wind (2-6 m/s)|669868    |3.41              |
|Warm (>20°C)           |Calm (<2 m/s)  

+----------+------+--------------+--------------+
|pm25_level|trips |observed_hours|trips_per_hour|
+----------+------+--------------+--------------+
|34.0      |4188  |1             |4188.0        |
|33.0      |9071  |2             |4535.5        |
|31.0      |5019  |1             |5019.0        |
|30.0      |18438 |3             |6146.0        |
|29.0      |18240 |2             |9120.0        |
|28.0      |17702 |3             |5900.67       |
|27.0      |14835 |2             |7417.5        |
|26.0      |66047 |10            |6604.7        |
|25.0      |18435 |3             |6145.0        |
|24.0      |54765 |10            |5476.5        |
|23.0      |20069 |5             |4013.8        |
|22.0      |54775 |12            |4564.58       |
|21.0      |80355 |17            |4726.76       |
|20.0      |83362 |16            |5210.13       |
|19.0      |79234 |17            |4660.82       |
|18.0      |132230|31            |4265.48       |
|17.0      |105092|29            |3623.86       |


+-----------------------------+-------+------+------+-------+-------------+
|pickup_zone                  |coldest|cool  |warm  |warmest|pct_variation|
+-----------------------------+-------+------+------+-------+-------------+
|Financial District South     |16.25  |13.0  |12.66 |10.69  |42.28        |
|Financial District North     |26.62  |21.65 |21.07 |18.24  |38.27        |
|Meatpacking/West Village West|49.81  |38.38 |40.34 |34.71  |37.0         |
|Lower East Side              |58.06  |45.25 |48.38 |41.47  |34.35        |
|Little Italy/NoLiTa          |52.69  |41.92 |43.93 |38.29  |32.57        |
|Greenwich Village South      |75.86  |62.62 |63.19 |57.27  |28.72        |
|West Village                 |121.51 |101.08|101.63|92.03  |28.33        |
|Battery Park City            |31.51  |28.26 |26.96 |23.97  |27.24        |
|TriBeCa/Civic Center         |66.51  |58.87 |57.36 |50.75  |27.0         |
|World Trade Center           |25.62  |21.19 |21.69 |19.84  |26.17        |
|East Villag

+-----------+---------+----------+-----------+---------+
|day_of_week|peak_hour|trip_count|active_days|avg_trips|
+-----------+---------+----------+-----------+---------+
|Monday     |18       |80482     |13         |6190.92  |
|Tuesday    |18       |99614     |13         |7662.62  |
|Wednesday  |18       |110914    |13         |8531.85  |
|Thursday   |18       |119124    |13         |9163.38  |
|Friday     |18       |102921    |13         |7917.0   |
|Saturday   |19       |96485     |13         |7421.92  |
|Sunday     |0        |79601     |13         |6123.15  |
+-----------+---------+----------+-----------+---------+


=== Q6 monthly trends ===


26/09/24 16:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 16:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 16:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----------+-----------+---------------+--------------------+------------------------+
|trip_month|total_trips|active_days|avg_daily_trips|prev_month_avg_daily|pct_change_vs_prev_month|
+----------+-----------+-----------+---------------+--------------------+------------------------+
|2024-01-01|2926910    |31         |94416.45       |NULL                |NULL                    |
|2024-02-01|2966705    |29         |102300.17      |94416.45            |8.35                    |
|2024-03-01|3523766    |31         |113669.87      |102300.17           |11.11                   |
|2024-04-01|2          |1          |2.0            |113669.87           |-100.0                  |
+----------+-----------+-----------+---------------+--------------------+------------------------+



26/09/24 16:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 16:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 16:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 16:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 16:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/24 16:06:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [4]:
spark.stop()